# BalePark's Agriculture & Climate SLM Project 

## Installation

In [1]:
!pip install -q "trl<0.12.0" "transformers<4.46.0" peft accelerate datasets
from transformers import set_seed

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 13.2 MB/s eta 0:00:00


## Load Data

In [2]:
from pathlib import Path
import pandas as pd  


# Variables 
COMPETITION_SLUG = "agriculture-climate-slm"
ON_KAGGLE = Path("/kaggle/input").exists()

def find_data_dir(slug: str) -> Path:
    """This function locate competitions CSVs under /kaggle/input. and is simply returning train_qa.csv dir"""
    if not ON_KAGGLE:
        return Path(".")
    root  = Path("/kaggle/input")
    for pattern in (
        f"competitions/agriculture-climate-slm",
        f"competitions/{slug.replace('-', '_')}",
        slug,
    ):
        candidate = root / pattern
        if (candidate / "train_qa.csv").exists():
            return candidate
    for path in root.rglob("train_qa.csv"):
        return path.parent
    raise FileNotFoundError(
        "Could not find train_qa.vsc. Join the competition and attach its dataset."

    )

DATA_DIR = find_data_dir(COMPETITION_SLUG)

OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Function to automatically locate files in Kaggle input
def find_file(filename: str) -> Path:
    root = Path("/kaggle/input")
    for path in root.rglob(filename):
        return path
    raise FileNotFoundError(f"Could not find {filename}. Please attach the dataset to the notebook.")

# Read the augmented dataset CSVs dynamically
new_train_qa = pd.read_csv(find_file("new_train_qa_4.csv"))
new_docs = pd.read_csv(find_file("new_documents_4.csv"))

# Combine documents
old_docs = pd.read_csv(DATA_DIR / "documents.csv")
docs = pd.concat([old_docs, new_docs], ignore_index=True)

# Combine QA training data
old_train = pd.read_csv(DATA_DIR / "train_qa.csv")
train = pd.concat([old_train, new_train_qa], ignore_index=True)

test = pd.read_csv(DATA_DIR/"test_questions.csv")

print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print(len(docs), "docs ·", len(train), "train ·", len(test), "test")
display(train.head(2))


Data dir: /kaggle/input/competitions/agriculture-climate-slm-challenge
Output dir: /kaggle/working
71 docs · 235 train · 12 test


,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2


## Model Detection

In [3]:
import os
# from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

model_candidates = []
for config_file in Path("/kaggle/input").rglob("config.json") if ON_KAGGLE else []:
    folder = config_file.parent
    
    has_tok = any((folder / fname).exists() for fname in ["tokenizer.json", "tokenizer_config.json"])
    has_weights = any(folder.glob("*.safetensors")) or any(folder.glob("*.bin")) or (folder / "model.safetensors.index.json").exists()
    
    if has_tok and has_weights:
        model_candidates.append(folder)

if ON_KAGGLE and model_candidates:
    MODEL_PATH = str(sorted(model_candidates, key=lambda p: len(str(p)))[0])
    print("Using attached Kaggle Model:", MODEL_PATH)
else:
    MODEL_PATH = "google/gemma-2-2b-it"  # local dev only — needs HF download
    print("Local/dev mode — will download from Hugging Face:", MODEL_PATH)


Using attached Kaggle Model: /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2


## Build prompt training & test 

In [4]:
import re

# Training prompt 
def build_prompt(row, docs_df, context_chars=None):
    doc = docs_df.loc[docs_df.document_id == row.document_id, "text"].iloc[0]
    context = doc if context_chars is None else doc[:context_chars]
    return (
        f"Crop: {row.crop} | Zone: {row.agro_zone} | Topic: {row.topic}\n"
        f"Question: {row.question}\n"
        f"Context: {context}\n"
        f"Answer:"
    )

# Retrieval helpers for test-time context matching
STOPWORDS = {"the","a","an","is","are","was","were","to","for","of","in","on","at","and","or",
             "my","i","how","what","when","should","do","does","did","it","this","that","with",
             "no","not","before","after","without","from","by","near","can","will","be","been"}

def tokenize(text):
    return set(w for w in re.findall(r"[a-z]+", str(text).lower())
               if w not in STOPWORDS and len(w) > 2)

# Precompute once — requires `docs` to already be the merged (old + new) dataframe
docs["_tokens"] = (docs["title"].astype(str) + " " + docs["text"].astype(str)).apply(tokenize)

def find_best_doc(row, docs_df):
    """Score every doc against a row's metadata + question text; return the best-matching document_id."""
    q_tokens = tokenize(row["question"])
    best_id, best_score = None, -1
    for _, d in docs_df.iterrows():
        score = 0
        if d["topic"] == row["topic"]:
            score += 100        # topic is the strongest, most reliable signal
        if d["crop"] == row["crop"]:
            score += 20         # crop match matters a lot
        if d["agro_zone"] == row["agro_zone"]:
            score += 10         # zone matters less than crop
        score += len(q_tokens & d["_tokens"])   # keyword overlap breaks ties
        if score > best_score:
            best_score, best_id = score, d["document_id"]
    return best_id

# Test prompt 
def build_prompt_for_test(t):
    doc_id = find_best_doc(t, docs)
    doc = docs.loc[docs.document_id == doc_id, "text"].iloc[0]
    return (
        f"Crop: {t.crop} | Zone: {t.agro_zone} | Topic: {t.topic}\n"
        f"Question: {t.question}\n"
        f"Context: {doc}\n"
        f"Answer:"
    )

In [5]:
# def build_prompt(row, docs_df, context_chars=None):
#     doc = docs_df.loc[docs_df.document_id == row.document_id, "text"].iloc[0]
#     context = doc if context_char is None else doc[:context_chars]
#     return (
#         f"Crop: {row.crop} | Zone: {row.agro_zone} | Topic: {row.topic}\n"
#         f"Question: {row.question}\n"
#         f"Context: {context}\n"
#         f"Answer:"
#     )

# def build_prompt_for_test(t):
#     sub = train[train["topic"] == t.topic]
#     row = sub.iloc[0] if len(sub) else train.iloc[0]
#     doc = docs.loc[docs.document_id == row.document_id, "text"].iloc[0]
#     return (
#         f"Crop: {t.crop} | Zone: {t.agro_zone} | Topic: {t.topic}\n"
#         f"Question: {t.question}\n"
#         f"Context: {context}\n"
#         f"Answer:"
#     )

## GPU Related

In [6]:
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))
# print(torch.cuda.device_count())

## Load Tokenizer + Model

In [7]:
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 96 # 48
LOCAL_ONLY = ON_KAGGLE

model = None
tok = None

def load_model_and_tokenizer():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=LOCAL_ONLY)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map= "auto",
        local_files_only=LOCAL_ONLY,
    )
    return tokenizer, base_model


def generate_output(active_model, active_tok, row):
    import torch

    prompt = build_prompt_for_test(row)
    inputs = active_tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH - MAX_NEW_TOKENS).to(active_model.device)
    with torch.no_grad():
        out = active_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.3,      # discourages verbatim repeats
            no_repeat_ngram_size=4,  
            pad_token_id=active_tok.eos_token_id,
        )
    decoded = active_tok.decode(out[0], skip_special_tokens=True)
    answer = decoded.split("Answer:")[-1].strip()
    for stop in ["\nQuestion:", "\nContext:", "\nCrop:"]:
        if stop in answer:
            answer = answer.split(stop)[0].strip()
    if ". " in answer:                             
        answer = answer.split(". ")[0].strip() + "."  
    return answer
    # return decoded.split("Answer:")[-1].strip()

tok, model = load_model_and_tokenizer()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Split training Dataset

In [8]:
# Split 
holdout_df = train.sample(frac=0.10, random_state=42)
train_fit_df = train.drop(holdout_df.index)


print(len(train_fit_df), "rows for training,", len(holdout_df), "rows held out for eval")

211 rows for training, 24 rows held out for eval


## Build SFT 

In [9]:
sft = [{"text": build_prompt(r, docs) + " " + r.reference_answer + tok.eos_token} for _, r in train_fit_df.iterrows()]
print(len(sft), "SFT rows")


211 SFT rows


## Get.Epoch N

In [10]:
# !pip install --upgrade torchao
# import gc
# import re
# import torch
# import pandas as pd
# from datasets import Dataset
# from peft import LoraConfig, TaskType, get_peft_model
# from trl import SFTConfig, SFTTrainer
# from transformers import set_seed   # FIX: was missing, set_seed() would NameError otherwise

# SEED = 42
# EPOCH_OPTIONS = [19, 20, 21]
# sweep_results = []

# def levenshtein(a, b):   # FIX: was undefined in this cell — now self-contained
#     if len(a) < len(b):
#         return levenshtein(b, a)
#     if len(b) == 0:
#         return len(a)
#     prev_row = range(len(b) + 1)
#     for i, ca in enumerate(a):
#         curr_row = [i + 1]
#         for j, cb in enumerate(b):
#             curr_row.append(min(
#                 prev_row[j + 1] + 1,      # deletion
#                 curr_row[j] + 1,          # insertion
#                 prev_row[j] + (ca != cb)  # substitution
#             ))
#         prev_row = curr_row
#     return prev_row[-1]

# def levenshtein_similarity(a, b):
#     str_a, str_b = str(a), str(b)
#     max_len = max(len(str_a), len(str_b))
#     if max_len == 0:
#         return 1.0
#     dist = levenshtein(str_a, str_b)
#     return 1.0 - (dist / max_len)

# def generate_for_train_row(active_model, active_tok, row):
#     prompt = build_prompt(row, docs)
#     inputs = active_tok(prompt, return_tensors="pt").to(active_model.device)
#     with torch.no_grad():
#         out = active_model.generate(
#             **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
#             repetition_penalty=1.3, no_repeat_ngram_size=4,
#             pad_token_id=active_tok.eos_token_id,
#         )
#     decoded = active_tok.decode(out[0], skip_special_tokens=True)
#     answer = decoded.split("Answer:")[-1].strip()
#     for stop in ["\nQuestion:", "\nContext:", "\nCrop:", "\nTopic:", "\n\n"]:
#         if stop in answer:
#             answer = answer.split(stop)[0].strip()
#     if ". " in answer:
#         answer = answer.split(". ")[0].strip() + "."
#     return answer

# for n_epochs in EPOCH_OPTIONS:
#     print(f"\n{'='*50}\nTraining with {n_epochs} epochs\n{'='*50}")

#     # Always start from a clean base model — no adapter-stacking across sweep iterations
#     tok, model = load_model_and_tokenizer()
#     set_seed(SEED)
#     model = get_peft_model(
#         model,
#         LoraConfig(
#             r=8, lora_alpha=16, lora_dropout=0.1,
#             # target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
#             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#             task_type=TaskType.CAUSAL_LM,
#         ),
#     )

#     SFTTrainer(
#         model=model,
#         args=SFTConfig(
#             report_to="none",
#             output_dir=str(OUTPUT_DIR / f"lora-sweep-{n_epochs}"),
#             num_train_epochs=n_epochs,
#             per_device_train_batch_size=2,
#             gradient_accumulation_steps=4,
#             learning_rate=3e-4,
#             warmup_ratio=0.1,
#             weight_decay=0.01,
#             logging_steps=50,
#             save_strategy="no",
#             max_seq_length=MAX_SEQ_LENGTH,
#             dataset_text_field="text",
#             fp16=torch.cuda.is_available(),
#             seed=SEED,   # FIX: fixes the trainer's internal batch shuffling too, for a fair sweep
#         ),
#         train_dataset=Dataset.from_dict({"text": [row["text"] for row in sft]}),
#         tokenizer=tok,
#     ).train()

#     # Evaluate on holdout — now tracking BOTH similarity and distance per row
#     sims, dists = [], []
#     for _, row in holdout_df.iterrows():
#         pred = generate_for_train_row(model, tok, row)
#         sims.append(levenshtein_similarity(pred, row["reference_answer"]))
#         dists.append(levenshtein(pred, row["reference_answer"]))

#     mean_sim = sum(sims) / len(sims)
#     mean_dist = sum(dists) / len(dists)
#     print(f"→ {n_epochs} epochs: mean similarity = {mean_sim*100:.2f}%, mean distance = {mean_dist:.2f}")

#     sweep_results.append({
#         "epochs": n_epochs,
#         "mean_similarity": mean_sim,
#         "mean_distance": mean_dist,
#     })

#     # Free GPU memory before the next iteration
#     del model, tok
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# sweep_df = pd.DataFrame(sweep_results).sort_values("mean_similarity", ascending=False)
# print("\n=== Sweep results ===")
# print(sweep_df.to_string(index=False))

# best_epochs = sweep_df.iloc[0]["epochs"]
# print(f"\nBest epoch count: {int(best_epochs)} "
#       f"(similarity={sweep_df.iloc[0]['mean_similarity']*100:.2f}%, "
#       f"distance={sweep_df.iloc[0]['mean_distance']:.2f})")

## Quick supervised fine-tune (full SFT)

In [11]:
# USE_SFT = False  # set True after attaching a model + enabling GPU
# SFT_TRAIN_MAX_ROWS = None  # e.g. 64 for a quick demo subset
# SFT_EPOCHS = 2



# if USE_SFT:
#     import torch
#     from datasets import Dataset
#     from trl import SFTConfig, SFTTrainer
    

#     # tok, model = load_model_and_tokenizer()
#     train_rows = sft[:SFT_TRAIN_MAX_ROWS] if SFT_TRAIN_MAX_ROWS else sft
#     print(f"Full SFT on {len(train_rows)} rows for {SFT_EPOCHS} epoch(s)")

#     SFTTrainer(
#         model=model,
#         args=SFTConfig(
#             report_to="none",
#             output_dir=str(OUTPUT_DIR / "sft-checkpoints"),
#             num_train_epochs=SFT_EPOCHS,
#             per_device_train_batch_size=1,
#             gradient_accumulation_steps=8,
#             learning_rate=2e-5,
#             logging_steps=5,
#             save_strategy="no", 
#             # max_length=MAX_SEQ_LENGTH,  
#             max_seq_length=MAX_SEQ_LENGTH,
#             dataset_text_field="text",
#             # fp16=torch.cuda.is_available(),
#             fp16=False
#         ),
#         train_dataset=Dataset.from_dict({"text": [row["text"] for row in train_rows]}),
#         # processing_class=tok,
#         tokenizer=tok,
#     ).train()

#     sft_submission = pd.DataFrame([
#         {"QuestionId": t["QuestionId"], "Answer": generate_output(model, tok, t)}
#         for _, t in test.iterrows()
#     ])
#     sft_submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
#     print("Wrote full SFT submission →", OUTPUT_DIR / "submission.csv")
# else:
#     print("Set USE_SFT=True for full supervised fine-tuning (§4b), or USE_LORA=True for LoRA (§4c).")


## Quick LoRA fine-tune

In [12]:
!pip install --upgrade torchao

USE_LORA = True  # set True after attaching a model + enabling GPU
LOCAL_ONLY = ON_KAGGLE

if USE_LORA:
    import torch
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from trl import SFTConfig, SFTTrainer

    # if model is None or tok is None:
    tok, model = load_model_and_tokenizer()

    set_seed(42)

    model = get_peft_model(
        model,
        LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.1,
            # target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            task_type=TaskType.CAUSAL_LM,
        ),
    )
    
    model.print_trainable_parameters()

    SFTTrainer(
        model=model,
        args=SFTConfig(
            report_to="none",
            output_dir=str(OUTPUT_DIR / "lora-checkpoints"),
            num_train_epochs=18,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate= 3e-4, #2e-4,
            warmup_ratio=0.1,
            weight_decay=0.01,
            logging_steps=5,
            save_strategy="no",
            max_seq_length=MAX_SEQ_LENGTH,
            # max_length=MAX_SEQ_LENGTH,
            dataset_text_field="text",
            fp16=torch.cuda.is_available(),
        ),
        train_dataset=Dataset.from_dict({"text": [row["text"] for row in sft]}),
        # processing_class=tok,
        tokenizer = tok,
    ).train()

    lora_submission = pd.DataFrame([
        {"QuestionId": t["QuestionId"], "Answer": generate_output(model, tok, t)}
        for _, t in test.iterrows()
    ])
    lora_submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print("Wrote LoRA submission →", OUTPUT_DIR / "submission.csv")
else:
    print("Set USE_LORA=True to run LoRA fine-tuning on top of §4b (or from the base model).")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 55.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 10,383,360 || all params: 2,624,725,248 || trainable%: 0.3956


Map:   0%|          | 0/211 [00:00<?, ? examples/s]

Step,Training Loss
5,3.239300
10,3.251100
15,2.605400
20,2.239100
25,1.991100
30,1.641200
35,1.686500
40,1.333900
45,1.131800
50,0.979500


The 'max_batch_size' argument of HybridCache is deprecated and will be removed in v4.46. Use the more precisely named 'batch_size' argument instead.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Wrote LoRA submission → /kaggle/working/submission.csv


## Lavenshtein Evaluation

In [13]:
import pandas as pd
import torch

def levenshtein(a, b):
    """Calculates raw Levenshtein distance."""
    if len(a) < len(b):
        return levenshtein(b, a)
    if len(b) == 0:
        return len(a)
    prev_row = range(len(b) + 1)
    for i, ca in enumerate(a):
        curr_row = [i + 1]
        for j, cb in enumerate(b):
            curr_row.append(min(
                prev_row[j + 1] + 1,
                curr_row[j] + 1,
                prev_row[j] + (ca != cb)
            ))
        prev_row = curr_row
    return prev_row[-1]

def levenshtein_similarity(a, b):
    """
    Computes normalized similarity ratio between 0.0 and 1.0.
    1.0 = exact match, 0.0 = completely different.
    """
    str_a, str_b = str(a), str(b)
    max_len = max(len(str_a), len(str_b))
    if max_len == 0:
        return 1.0
    
    dist = levenshtein(str_a, str_b)
    return 1.0 - (dist / max_len)

def generate_for_train_row(active_model, active_tok, row):
    prompt = build_prompt(row, docs)
    inputs = active_tok(prompt, return_tensors="pt").to(active_model.device)
    with torch.no_grad():
        out = active_model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            repetition_penalty=1.3, no_repeat_ngram_size=4,
            pad_token_id=active_tok.eos_token_id,
        )
    decoded = active_tok.decode(out[0], skip_special_tokens=True)
    answer = decoded.split("Answer:")[-1].strip()
    for stop in ["\nQuestion:", "\nContext:", "\nCrop:", "\nTopic:", "\n\n"]:
        if stop in answer:
            answer = answer.split(stop)[0].strip()
    if ". " in answer:                              # NEW
        answer = answer.split(". ")[0].strip() + "."  # NEW
    return answer

# Run evaluation loop
results = []
for _, row in holdout_df.iterrows():
    pred = generate_for_train_row(model, tok, row)
    ref = row["reference_answer"]
    
    # Keep 'dist' intact for upstream compatibility
    dist = levenshtein(pred, ref)
    sim = levenshtein_similarity(pred, ref)
    
    results.append({
        "QuestionId": row["QuestionId"],
        "prediction": pred,
        "reference": ref,
        "distance": dist,
        "similarity": sim
    })

results_df = pd.DataFrame(results)

print(f"Mean holdout Levenshtein distance  : {results_df['distance'].mean():.2f}")
print(f"Mean holdout Levenshtein similarity: {results_df['similarity'].mean() * 100:.2f}%")

results_df

Mean holdout Levenshtein distance  : 51.42
Mean holdout Levenshtein similarity: 38.77%


,QuestionId,prediction,reference,distance,similarity
0,70,Historically only in East Africa and Southern ...,Historically it was reported only in cassava-g...,63,0.307692
1,253,"Check for leaf hopper feeding, which can withe...",Possibly leaf hopper feeding; late-sown maize ...,72,0.351351
2,226,About three percent.,Ninety-four percent of sampled fields relied e...,53,0.196970
3,10,"It should be dark brown or black, crumbly like...","It should be dark, crumbly, and free of undeco...",42,0.447368
4,148,"Yes, droughts can intensify conflict over wate...","Yes, it can intensify conflict between farmers...",43,0.542553
5,110,"Yes, there was variation from year to year; so...","Yes, the reduction depends on how much aflatox...",76,0.315315
6,242,Likely the legume pod borer; larvae bore into ...,"Likely the legume pod borer (Maruca vitrata), ...",42,0.517241
7,94,"No, varieties have different tolerances; choos...","No, varieties differ in tolerance to major dis...",45,0.476744
8,269,One kilogram of wood ashes mixed with forty ki...,One kilogram of wood ash per forty kilograms o...,19,0.688525
9,16,Build a tank and use first-flush diverter to r...,Channel roof runoff into screened tanks with f...,56,0.176471
